## Initialize

In [ ]:
%matplotlib inline

import gzip
import matplotlib.pyplot as plt
import matplotlib
import os 
import numpy as np
import struct
import random

import keras 
from keras.models import load_model
from keras.models import Sequential
from keras.layers import Dense 
from keras.callbacks import TensorBoard
from keras import backend



In [ ]:
base_path = '/Users/lime/src/ml/ocr/'
dataset_path = base_path + 'dataset/gzip/'

## Load the EMNIST dataset

In [ ]:
# Extract the dataset from zip files 
def read_idx(filename):

    print('Processing data from %s.' % filename)
    with gzip.open(filename, 'rb') as f:
        # read the "magic number" first
        z, dtype, dim = struct.unpack('>HBB', f.read(4))
        print("Dimensions:", dim)
        # get the shape (size in each dimension) of the data
        shape = tuple(struct.unpack('>I', f.read(4))[0] for d in range(dim))
        print("Shape:", shape)
        # return the data as a reshaped numpy array
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(shape)

In [ ]:
# The dataset coulb be loaded from https://www.nist.gov/itl/products-and-services/emnist-dataset
def load_emnist():
    
    train_images = dataset_path + 'emnist-byclass-train-images-idx3-ubyte.gz'
    train_labels = dataset_path + 'emnist-byclass-train-labels-idx1-ubyte.gz'
    test_images = dataset_path + 'emnist-byclass-test-images-idx3-ubyte.gz'
    test_labels = dataset_path + 'emnist-byclass-test-labels-idx1-ubyte.gz'

    train_X = read_idx(train_images)
    train_y = read_idx(train_labels)
    test_X = read_idx(test_images)
    test_y = read_idx(test_labels)

    return (train_X, train_y, test_X, test_y)

In [ ]:
raw_train_X, raw_train_y, raw_test_X, raw_test_y = load_emnist();

## Visualization of images from dataset

In [ ]:
plt.imshow(raw_train_X[100].T, cmap='gray')
plt.colorbar()
plt.show()

In [ ]:
matplotlib.use('macosx')  # or 'macosx'
range_of_10 = [random.random() for _ in range(10)]

# Label (category) Mapping:
labels = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'

# Scrolling visualizer
fig, ax = plt.subplots()
for x in range(len(range_of_10)):
    ax.clear()
    ax.imshow([i for i in 255 - raw_train_X[x].T], cmap='gray')
    title = 'label = %d = %s' % (raw_train_y[x], labels[raw_train_y[x]])
    ax.set_title(title, fontsize=20)
    plt.pause(1)

# Data shapes

In [ ]:
print('train_X shape: ',raw_train_X.shape)
print('train_y shape: ',raw_train_y.shape)
print('test_X shape: ',raw_test_X.shape)
print('test_y shape: ',raw_test_y.shape)

In [ ]:
# Save the dimensions of the input images
img_height = len(raw_train_X[0])
img_width = len(raw_train_X[1])
input_shape = img_height*img_width

print('Original Image Dimensions: ', img_height, img_width)
print('Flattened image length: ', input_shape)

In [ ]:
train_X = raw_train_X.reshape(len(raw_train_X), input_shape)
test_X = raw_test_X.reshape(len(raw_test_X), input_shape)

print('Training dataset shape: ', train_X.shape)
print('Model input shape: ', input_shape)

# Data normalization

### Change the type of data and the value of gray scale. Before -> 0 - 255, after -> 0 - 1 

In [ ]:
# Values start as integers:
print(train_X.dtype)

# Set as float32's
train_X = train_X.astype('float32')
test_X = test_X.astype('float32')

# Normalize! 
train_X /= 255
test_X /= 255

print('Flattened length of numpy data array: ',img_height*img_width)

### Display iamge

In [ ]:
%matplotlib inline 

plt.imshow(train_X.reshape(len(train_X),28,28)[100].T, cmap='gray')
plt.colorbar()
plt.show()

In [ ]:
print(raw_train_y)

label_vec = labels.split()
n_cat = len(labels)



In [ ]:
train_y = keras.utils.to_categorical(raw_train_y)
test_y = keras.utils.to_categorical(raw_test_y)

print(raw_train_y[5], train_y[5])



# Model building

## Define model

In [ ]:
model = keras.models.Sequential()
model.add(Dense(16, input_dim=input_shape, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(20, activation='relu'))
model.add(Dense(n_cat, activation='softmax'))

## Compile model

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

## Train model

In [ ]:
print(len(train_X))
print(len(train_y))

In [ ]:
# checkpoint
log_path = base_path + 'logs/'
# tensorboard = Tensorboard(filepath, monitor='val_acc', verbose=1, save_best_only=False, mode='max')
tensorboard = TensorBoard(log_dir=log_path, histogram_freq=0, write_graph=True)
callbacks_list = [tensorboard]

In [ ]:
model.fit(train_X, train_y, epochs=3,  callbacks=callbacks_list)

model.save('model002_epoch3.h5')


## Model validation

In [ ]:
results = model.evaluate(test_X, test_y)
print(print("\nLoss: %.2f%%, Accuracy: %.2f%%" % (results[0], results[1])))

## Test the model

In [ ]:
%matplotlib inline

# Select a 'random' subset
random.seed(112358)
sample = np.arange(raw_test_X.shape[0])
np.random.shuffle(sample)
sample = sample[0:10]

# Format for input into network
results = np.round(model.predict(test_X[sample], verbose=1), decimals=2)
resultLabels = np.argmax(results, axis=1)

# plt.imshow(raw_train_X[55].T, cmap='gray')

fig=plt.figure(figsize=(15, 8))
for i in range(10):
    fig.add_subplot(2, 5, i+1, aspect='equal')
    plt.imshow(raw_test_X[sample[i]].T, cmap='gray')
    plt.title('This image is  {}'.format(labels[resultLabels[i]]))
    plt.xlabel("Img {}".format(sample[i]))

## Launch TensorBoard

In [ ]:
%%bash
tensorboard --logdir=/Users/lime/src/ml/ocr/logs/